# 🔬 Data exploration and preparation
In this notebook, we'll examine the dataset and create a subset of it for further analysis. The dataset was relatively clean when downloaded, though we addressed some problematic delimiter issues for you. If you're interested in tackling these issues firsthand, the original dataset is available at the [Book-Crossing Dataset](http://www2.informatik.uni-freiburg.de/~cziegler/BX/).

### 1. Loading the data
Load the three datasets and explore the data.


In [5]:
import pandas as pd
import numpy as np

# Load the three datasets
books = pd.read_csv("data/BX-Books.csv", encoding='latin-1', sep=';', quotechar='"', on_bad_lines='skip')
ratings = pd.read_csv("data/BX-Book-Ratings.csv", encoding='latin-1', sep=';')
users = pd.read_csv("data/BX-Users.csv", encoding='latin-1', sep=';', on_bad_lines='skip')

# Display basic information about each dataset
print("Books Dataset Shape:", books.shape)
print("\nBooks Columns:", books.columns.tolist())
print("\nBooks First 5 rows:")
print(books.head())

print("\n" + "="*80 + "\n")

print("Ratings Dataset Shape:", ratings.shape)
print("\nRatings Columns:", ratings.columns.tolist())
print("\nRatings First 5 rows:")
print(ratings.head())

print("\n" + "="*80 + "\n")

print("Users Dataset Shape:", users.shape)
print("\nUsers Columns:", users.columns.tolist())
print("\nUsers First 5 rows:")
print(users.head())

C:\Users\Dora Drogeanu\AppData\Local\Temp\ipykernel_2620\1305009736.py:5: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv("data/BX-Books.csv", encoding='latin-1', sep=';', quotechar='"', on_bad_lines='skip')


Books Dataset Shape: (271379, 8)

Books Columns: ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']

Books First 5 rows:
         ISBN                                         Book-Title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

            Book-Author Year-Of-Publication                Publisher  \
0    Mark P. O. Morford                2002  Oxford University Press   
1  Richard Bruce Wright                2001    HarperFlamingo Canada   
2          Carlo D'Este                1991          HarperPerennial   
3      Gina Bari Kolata                1999     Farrar Straus Giroux   
4       E. J. W. Barber                199

### 2. Cleaning the data
Ensure that all reviews are linked to a book. Investigate whether there are any reviews that lack a corresponding book or user. Verify the accuracy of author names and identify any anomalies, such as users who have submitted an unusually high number of reviews. Describe the process you followed to clean the data.

In [8]:
# Step 1: Check for missing values in all datasets

print("MISSING VALUES ANALYSIS")

print("\nBooks - Missing values:")
print(books.isnull().sum())
print(f"\nTotal rows with any missing values: {books.isnull().any(axis=1).sum()}")


print("\nRatings - Missing values:")
print(ratings.isnull().sum())
print(f"\nTotal rows with any missing values: {ratings.isnull().any(axis=1).sum()}")


print("\nUsers - Missing values:")
print(users.isnull().sum())
print(f"\nTotal rows with any missing values: {users.isnull().any(axis=1).sum()}")

# Step 2: Check for reviews without corresponding books

print("CHECKING REFERENTIAL INTEGRITY - Reviews to Books")

# Get unique ISBN values
books_isbn = set(books['ISBN'].astype(str).unique())
ratings_isbn = set(ratings['ISBN'].astype(str).unique())

# Find reviews with books not in the books dataset
reviews_without_books = ratings_isbn - books_isbn
print(f"\nNumber of unique ISBNs in ratings: {len(ratings_isbn)}")
print(f"Number of unique ISBNs in books: {len(books_isbn)}")
print(f"Reviews with books NOT in books dataset: {len(reviews_without_books)}")

if len(reviews_without_books) > 0:
    missing_book_count = ratings[ratings['ISBN'].isin(reviews_without_books)].shape[0]
    print(f"Number of ratings referring to missing books: {missing_book_count}")
    print(f"Percentage of ratings affected: {100 * missing_book_count / len(ratings):.2f}%")

# Step 3: Check for reviews without corresponding users

print("CHECKING REFERENTIAL INTEGRITY - Reviews to Users")


# Get unique User-ID values
users_id = set(users['User-ID'].astype(str).unique())
ratings_user_id = set(ratings['User-ID'].astype(str).unique())

# Find reviews with users not in the users dataset
reviews_without_users = ratings_user_id - users_id
print(f"\nNumber of unique User-IDs in ratings: {len(ratings_user_id)}")
print(f"Number of unique User-IDs in users: {len(users_id)}")
print(f"Reviews with users NOT in users dataset: {len(reviews_without_users)}")

if len(reviews_without_users) > 0:
    missing_user_count = ratings[ratings['User-ID'].isin(reviews_without_users)].shape[0]
    print(f"Number of ratings from missing users: {missing_user_count}")
    print(f"Percentage of ratings affected: {100 * missing_user_count / len(ratings):.2f}%")

# Step 4: Author name analysis

print("AUTHOR NAME ANALYSIS")


print(f"\nTotal unique authors: {books['Book-Author'].nunique()}")
print(f"\nAuthors with missing values: {books['Book-Author'].isnull().sum()}")
print(f"\nTop 15 most common authors:")
print(books['Book-Author'].value_counts().head(15))

# Check for unusual author names - books with missing author info
print("\n\nBooks with missing author information:")
missing_authors = books[books['Book-Author'].isnull()]
print(f"Count: {len(missing_authors)}")
if len(missing_authors) > 0:
    print("\nExamples of books with missing authors:")
    print(missing_authors[['ISBN', 'Book-Title', 'Book-Author']].head())

# Step 5: Identify anomalies - Users with unusually high number of reviews
print("\n" + "=" * 80)
print("USER ACTIVITY ANOMALIES")
print("=" * 80)

user_review_counts = ratings['User-ID'].value_counts()
print(f"\nUser review statistics:")
print(f"  Mean reviews per user: {user_review_counts.mean():.2f}")
print(f"  Median reviews per user: {user_review_counts.median():.2f}")
print(f"  Std Dev: {user_review_counts.std():.2f}")
print(f"  Min: {user_review_counts.min()}")
print(f"  Max: {user_review_counts.max()}")
print(f"  Q3 (75th percentile): {user_review_counts.quantile(0.75):.0f}")

# Identify outliers (users with unusually high activity)
q3 = user_review_counts.quantile(0.75)
iqr = user_review_counts.quantile(0.75) - user_review_counts.quantile(0.25)
outlier_threshold = q3 + 1.5 * iqr

print(f"\nOutlier threshold (Q3 + 1.5*IQR): {outlier_threshold:.0f}")
outlier_users = user_review_counts[user_review_counts > outlier_threshold]
print(f"Number of power users (outliers): {len(outlier_users)}")
print(f"Percentage of users: {100 * len(outlier_users) / len(user_review_counts):.2f}%")
print(f"Percentage of all ratings from these users: {100 * outlier_users.sum() / len(ratings):.2f}%")

print("\n\nTop 20 most active users:")
print(user_review_counts.head(20))

# Step 6: Book rating statistics
print("\n" + "=" * 80)
print("BOOK RATING DISTRIBUTION")
print("=" * 80)

print(f"\nRating distribution:")
print(ratings['Book-Rating'].value_counts().sort_index())

implicit_ratings = (ratings['Book-Rating'] == 0).sum()
explicit_ratings = (ratings['Book-Rating'] > 0).sum()

print(f"\nImplicit ratings (0): {implicit_ratings} ({100*implicit_ratings/len(ratings):.2f}%)")
print(f"Explicit ratings (1-10): {explicit_ratings} ({100*explicit_ratings/len(ratings):.2f}%)")

# Step 7: Summary of cleaning recommendations
print("\n" + "=" * 80)
print("DATA CLEANING SUMMARY & RECOMMENDATIONS")
print("=" * 80)

print("""
Based on the analysis above, here are the key findings and recommendations:

1. **Referential Integrity**: Check whether reviews without corresponding books/users should be removed

2. **Missing Data**: Address any missing author information in the books dataset

3. **Implicit Ratings**: Decide whether to exclude implicit (0-rated) reviews before subsetting

4. **Power Users**: Consider the impact of users with extremely high review counts on the dataset

5. **Next Steps**: Apply these findings when subsetting the data according to the publication criteria
""")

MISSING VALUES ANALYSIS

Books - Missing values:
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

Total rows with any missing values: 7

Ratings - Missing values:
User-ID        0
ISBN           0
Book-Rating    0
dtype: int64

Total rows with any missing values: 0

Users - Missing values:
User-ID          0
Location         0
Age         110762
dtype: int64

Total rows with any missing values: 110762
CHECKING REFERENTIAL INTEGRITY - Reviews to Books

Number of unique ISBNs in ratings: 340555
Number of unique ISBNs in books: 271379
Reviews with books NOT in books dataset: 70385
Number of ratings referring to missing books: 118604
Percentage of ratings affected: 10.32%
CHECKING REFERENTIAL INTEGRITY - Reviews to Users

Number of unique User-IDs in ratings: 105283
Number of unique User-IDs in users: 278858
Reviews with users 

### 3. Subsetting the data
The publication accompanied with this dataset [Improving Recommendation Lists Through Topic Diversification](http://www2.informatik.uni-freiburg.de/~cziegler/BX/WWW-2005-Preprint.pdf) by Cai-Nicolas Ziegler, Sean M. McNee, Joseph A. Konstan, Georg Lausen; describes the process of subsetting (condensation steps) the dataset as follows (p5): 

> Hence, we discarded all books missing taxonomic descriptions, along with all ratings referring to them. Next, we also removed book titles with fewer than 20 overall mentions. Only community members with at least five ratings each were kept. 

Investigate the significance of these parameters for the dataset as a whole. Additionally, decide whether to include implicit ratings (where Book-Rating equals 0) in your final dataset. Consider the potential consequences of this choice. Would you opt to exclude them prior to assessing other parameters, or would it be more appropriate to exclude them later?

Although the publication outlines the expected dimensions of the resulting dataset, it's acceptable if your findings vary at this stage.

In [9]:
# code goes here


# this combination worked pretty well 
ratings = ratings[ratings['Book-Rating'] != 0]

x = ratings['ISBN'].value_counts() >= 20
idx = x[x].index
ratings = ratings[ratings['ISBN'].isin(idx)]

x = ratings['User-ID'].value_counts() >= 10
idx = x[x].index
ratings_final = ratings[ratings['User-ID'].isin(idx)]
ratings_final.shape

(41337, 3)

### 4. Extra step
Examine the `BX-Books.csv` file specifically for the book Robots and _Empire by Isaac Asimov_. Identify any issues you come across. Would you address these issues?

Given that this could pose a problem for our dataset, consider how you would resolve it. You may need to revisit step 2 if you decide to undertake this additional step.

In [ ]:
# Let's search more broadly for "Robots" titles by Asimov


# Search 1: Direct match (already done above)
print("\n1. Direct Match Search:")
print("   Query: Book-Title == 'Robots and Empire' AND Book-Author == 'Isaac Asimov'")
result = books[(books['Book-Title'] == 'Robots and Empire') & (books['Book-Author'] == 'Isaac Asimov')]
print(f"   Found {len(result)} records")
if len(result) > 0:
    print("\n   Results:")
    for idx, row in result.iterrows():
        print(f"   ISBN: {row['ISBN']}, Year: {row['Year-Of-Publication']}, Publisher: {row['Publisher']}")

# Search 2: Look for any Asimov books with "Robots" in title
print("\n2. Broader Search - All Asimov books with 'Robots' in title:")
asimov_robots = books[
    (books['Book-Author'].str.contains('Asimov', case=False, na=False)) &
    (books['Book-Title'].str.contains('Robots', case=False, na=False))
]
print(f"   Found {len(asimov_robots)} records")
print(asimov_robots[['ISBN', 'Book-Title', 'Book-Author']].to_string())

# Search 3: Look for titles with special characters or encoding issues (underscore)
print("\n\n3. Searching for underscore or encoding issues:")
underscore_titles = books[books['Book-Title'].str.contains('_', na=False)]
print(f"   Books with underscores in title: {len(underscore_titles)}")
if len(underscore_titles) > 0:
    print("\n   Examples:")
    print(underscore_titles[['ISBN', 'Book-Title', 'Book-Author']].head(10).to_string())

# Search 4: Look for potential delimiter issues - titles that might have been split incorrectly
print("\n\n4. Checking for potential delimiter/parsing issues:")
print("   Looking for very long ISBNs or unusual field lengths...")

# Check field lengths
print(f"\n   ISBN length statistics:")
books['isbn_length'] = books['ISBN'].astype(str).str.len()
print(f"   Min: {books['isbn_length'].min()}, Max: {books['isbn_length'].max()}, Mean: {books['isbn_length'].mean():.2f}")

# Find unusual ISBNs
unusual_isbns = books[books['isbn_length'] > 20]
if len(unusual_isbns) > 0:
    print(f"\n   Found {len(unusual_isbns)} records with unusually long ISBNs (potential parsing issues):")
    print(unusual_isbns[['ISBN', 'Book-Title', 'Book-Author']].head().to_string())

# Search 5: Get detailed view of the three editions found
print("\n\n5. DETAILED ANALYSIS OF THE THREE EDITIONS:")
print("="*80)
for idx, row in result.iterrows():
    print(f"\nEdition - ISBN: {row['ISBN']}")
    print(f"  Title: {row['Book-Title']}")
    print(f"  Author: {row['Book-Author']}")
    print(f"  Year: {row['Year-Of-Publication']}")
    print(f"  Publisher: {row['Publisher']}")
    
    # Check if this ISBN has ratings
    isbn_ratings = ratings[ratings['ISBN'] == row['ISBN']]
    print(f"  Number of ratings for this ISBN: {len(isbn_ratings)}")
    if len(isbn_ratings) > 0:
        print(f"    Rating distribution: {isbn_ratings['Book-Rating'].value_counts().sort_index().to_dict()}")

print("\n" + "="*80)
print("ISSUE SUMMARY:")
print("="*80)

if len(result) > 1:
    print(f"""
✓ IDENTIFIED ISSUE: Duplicate Editions
  - Found {len(result)} different editions of 'Robots and Empire' by Isaac Asimov
  - Different ISBNs: {result['ISBN'].tolist()}
  - Published in different years: {result['Year-Of-Publication'].tolist()}
  
POTENTIAL PROBLEMS:
  1. These represent the SAME book but DIFFERENT editions (hardcover vs paperback)
  2. When building a recommendation system, we may want to treat all editions as the same book
  3. Or conversely, users may rate one edition differently from another
  4. Duplicates could skew popularity metrics if not handled carefully
  
RESOLUTION OPTIONS:
  1. Merge all editions under one ISBN (canonical ISBN)
  2. Keep separate but track edition relationships
  3. Use ISBN to identify and handle duplicates during subsetting
""")
elif len(result) == 1:
    print("\n✓ NO ISSUES FOUND: Single entry for this book")
else:
    print("\n✗ NOT FOUND: Book not found in dataset")


SEARCHING FOR ROBOTS AND EMPIRE BY ISAAC ASIMOV - ISSUE INVESTIGATION

1. Direct Match Search:
   Query: Book-Title == 'Robots and Empire' AND Book-Author == 'Isaac Asimov'
   Found 3 records

   Results:
   ISBN: 0586062009, Year: 1986, Publisher: HarperCollins
   ISBN: 0345328949, Year: 1991, Publisher: Del Rey Books
   ISBN: 0385190921, Year: 1985, Publisher: Smithmark Pub

2. Broader Search - All Asimov books with 'Robots' in title:
   Found 15 records
              ISBN                                                                                               Book-Title   Book-Author
13873   0345315715                                                                          Robots of Dawn (Robots of Dawn)  Isaac Asimov
19462   0586025944                                                                                   The Rest of the Robots  Isaac Asimov
19463   0586062009                                                                                        Robots and Empire  

In [17]:
# RESOLUTION: Handling duplicate book editions
print("="*80)
print("ADDRESSING THE DUPLICATE EDITIONS ISSUE")
print("="*80)

# Fix data type issue with Year-Of-Publication
books['Year-Of-Publication'] = pd.to_numeric(books['Year-Of-Publication'], errors='coerce')

print("""
ISSUE IDENTIFIED:
  "Robots and Empire" by Isaac Asimov exists in multiple editions with different ISBNs
  This can bias our recommendation system because:
  - Similar users might rate different physical editions the same
  - Popularity metrics count separate sales, not unique content
  - Creates sparse rating matrix issues

SOLUTION APPROACH:
  Option 1 (SIMPLE): Filter out duplicate books during subsetting
  Option 2 (BETTER): Create a book-to-canonical-edition mapping
  Option 3 (BEST): Merge ratings across editions for collaborative filtering
""")

print("\nIMPLEMENTATION: Option 2 - Create Edition Mapping\n")

# Find all books with the same title and author (duplicates/editions)
book_groups = books.groupby(['Book-Title', 'Book-Author']).size()
duplicate_books = book_groups[book_groups > 1]

print(f"Total books with multiple editions: {len(duplicate_books)}")
print(f"\nTop 20 most common books with multiple editions:")
print(duplicate_books.sort_values(ascending=False).head(20))

# Create a mapping of all ISBNs to canonical ISBN (first/earliest edition)
isbn_to_canonical = {}

for (title, author), group_df in books.groupby(['Book-Title', 'Book-Author']):
    if len(group_df) > 1:
        # Use the earliest published edition as canonical
        canonical_isbn = group_df.sort_values('Year-Of-Publication').iloc[0]['ISBN']
        for isbn in group_df['ISBN'].values:
            isbn_to_canonical[isbn] = canonical_isbn

print(f"\nCreated mapping for {len(isbn_to_canonical)} ISBNs")

# Show the mapping for Robots and Empire
print(f"\nExample - Robots and Empire editions mapping:")
robots_empire = books[(books['Book-Title'] == 'Robots and Empire') & (books['Book-Author'] == 'Isaac Asimov')]
for isbn in robots_empire['ISBN'].values:
    print(f"  {isbn} → {isbn_to_canonical.get(isbn, isbn)}")

# Apply the canonical ISBN mapping to ratings
print(f"\nApplying canonical ISBN mapping to ratings...")
ratings_copy = ratings.copy()
ratings_copy['ISBN_canonical'] = ratings_copy['ISBN'].map(isbn_to_canonical).fillna(ratings_copy['ISBN'])

print(f"Before mapping: {ratings_copy['ISBN'].nunique()} unique ISBNs in ratings")
print(f"After mapping: {ratings_copy['ISBN_canonical'].nunique()} unique ISBNs")

# Show the impact on Robots and Empire
robots_empire_isbns = robots_empire['ISBN'].tolist()
robots_ratings_before = ratings[ratings['ISBN'].isin(robots_empire_isbns)]
robots_ratings_after = ratings_copy[ratings_copy['ISBN_canonical'].isin(robots_empire_isbns)]

print(f"\nRobots and Empire ratings impact:")
print(f"  Before consolidation: {len(robots_ratings_before)} ratings across 3 editions")
print(f"  After consolidation: {len(robots_ratings_after)} ratings under canonical ISBN")

print(f"\n✓ RECOMMENDATION: Use the canonical ISBN mapping when building the dataset")
print(f"  This will help ensure recommendation quality by consolidating edition variants.")

ADDRESSING THE DUPLICATE EDITIONS ISSUE

ISSUE IDENTIFIED:
  "Robots and Empire" by Isaac Asimov exists in multiple editions with different ISBNs
  This can bias our recommendation system because:
  - Similar users might rate different physical editions the same
  - Popularity metrics count separate sales, not unique content
  - Creates sparse rating matrix issues

SOLUTION APPROACH:
  Option 1 (SIMPLE): Filter out duplicate books during subsetting
  Option 2 (BETTER): Create a book-to-canonical-edition mapping
  Option 3 (BEST): Merge ratings across editions for collaborative filtering


IMPLEMENTATION: Option 2 - Create Edition Mapping

Total books with multiple editions: 15746

Top 20 most common books with multiple editions:
Book-Title                                                       Book-Author                
Little Women                                                     Louisa May Alcott              21
Wuthering Heights                                                Emil

### 5. Save the new dataset(s)
Save the dataset(s) in distinct named CSV-files for later usage. Move the file(s) to the data directory.


In [19]:
import os
os.makedirs("data", exist_ok=True)

# Filter to final datasets
books_final = books[books['ISBN'].isin(ratings_final['ISBN'])].copy()
users_final = users[users['User-ID'].isin(ratings_final['User-ID'])].copy()

# Save cleaned datasets
ratings_final.to_csv("data/BX-Book-Ratings-Cleaned.csv", sep=';', encoding='utf-8', index=False)
books_final.to_csv("data/BX-Books-Cleaned.csv", sep=';', encoding='utf-8', index=False)
users_final.to_csv("data/BX-Users-Cleaned.csv", sep=';', encoding='utf-8', index=False)

# Save ISBN mapping
canonical_mapping_df = pd.DataFrame({'ISBN': list(isbn_to_canonical.keys()), 
                                      'Canonical_ISBN': list(isbn_to_canonical.values())})
canonical_mapping_df.to_csv("data/ISBN-Canonical-Mapping.csv", sep=';', encoding='utf-8', index=False)

print("✓ Datasets saved successfully!")
print(f"  Ratings: {ratings_final.shape}")
print(f"  Books: {books_final.shape}")
print(f"  Users: {users_final.shape}")
print(f"  ISBN Mappings: {len(canonical_mapping_df)}")

✓ Datasets saved successfully!
  Ratings: (41337, 3)
  Books: (2124, 9)
  Users: (1834, 3)
  ISBN Mappings: 35921
